In [ ]:
"""Imports."""

import numpy as np
import pandas as pd
from matplotlib import pyplot as plt
from sklearn import linear_model as sklearn_linear_model

%matplotlib widget

In [ ]:
"""Functions for 3-D geometry computations."""


def _dot_row(x, y):
    return np.sum(x * y, axis=1, keepdims=True)


def nearest_focus(p_0, v_0, p_1, v_1):
    """Find the nearest point to two 3-dimensional lines.

    The lines are defined by:
        line 0 spanned by p_0 + t d_0
        line 1 spanned by p_1 + t d_1

    The nearest point to these two 3-dimensional lines can be computed with
    linear algebra.
    """
    cross = np.cross(v_0, v_1)
    perp_0 = np.cross(v_0, cross)
    perp_1 = np.cross(v_1, cross)
    coef_0 = _dot_row(p_1 - p_0, perp_1) / _dot_row(v_0, perp_1)
    point_0 = p_0 + coef_0 * v_0
    coef_1 = _dot_row(p_0 - p_1, perp_0) / _dot_row(v_1, perp_0)
    point_1 = p_1 + coef_1 * v_1
    midpoint = 0.5 * (point_0 + point_1)

    return point_0, point_1, midpoint


def test_nearest_focus():
    p_0 = np.array([[1.0, 2.0, 3.0], [10.0, 21.0, 3.0]])
    v_0 = np.array([[1.0, 2.0, 3.0], [1.0, 2.0, 3.0]])
    p_1 = np.array([[4.0, 2.0, 3.0], [4.0, 2.0, 3.0]])
    v_1 = np.array([[4.0, 2.0, 5.0], [4.0, -2.0, 5.0]])

    point_0, point_1, midpoint = nearest_focus(p_0, v_0, p_1, v_1)

    pv_0 = p_0 + v_0
    pv_1 = p_1 + v_1
    num_points = midpoint.shape[0]
    for i in range(num_points):
        fig = plt.figure(figsize=(3, 3))
        ax = fig.add_subplot(111, projection="3d")
        ax.scatter(p_0[i, 0], p_0[i, 1], p_0[i, 2], c="b")
        ax.scatter(pv_0[i, 0], pv_0[i, 1], pv_0[i, 2], c="b")
        ax.scatter(point_0[i, 0], point_0[i, 1], point_0[i, 2], c="b", s=100)
        ax.scatter(p_1[i, 0], p_1[i, 1], p_1[i, 2], c="r")
        ax.scatter(pv_1[i, 0], pv_1[i, 1], pv_1[i, 2], c="r")
        ax.scatter(point_1[i, 0], point_1[i, 1], point_1[i, 2], c="r", s=100)
        ax.scatter(
            midpoint[i, 0], midpoint[i, 1], midpoint[i, 2], c="k", s=100
        )


def _compute_focus_points(
    right_eye, left_eye, retina_z, theta, pupil_sep, inward_skew
):
    left_eye = np.copy(left_eye)
    right_eye = np.copy(right_eye)

    # Translate left eye
    delta_left_right = pupil_sep * np.array([np.sin(theta), np.cos(theta)])
    left_eye += delta_left_right

    # Get retina centers
    retina_right = np.array([0.0, 0.0, retina_z])
    retina_left = np.concatenate([delta_left_right, [retina_z]])

    # Skew eyes to compute pupil centers
    skew_vector = delta_left_right * inward_skew
    pupil_right = right_eye + skew_vector
    pupil_left = left_eye - skew_vector
    pupil_right = np.concatenate(
        [pupil_right, np.zeros((pupil_right.shape[0], 1))], axis=1
    )
    pupil_left = np.concatenate(
        [pupil_left, np.zeros((pupil_left.shape[0], 1))], axis=1
    )

    # Compute focus points
    vector_right = pupil_right - retina_right[None]
    vector_left = pupil_left - retina_left[None]
    point_right, point_left, midpoint = nearest_focus(
        retina_right, vector_right, retina_left, vector_left
    )

    # Compute goodness-of-fit
    point_distance = np.linalg.norm(point_left - point_right, axis=1)
    retina_midpoint = 0.5 * (retina_right + retina_left)
    retina_distance = np.linalg.norm(midpoint - retina_midpoint[None], axis=1)
    goodness_of_fit = np.nanmean(point_distance / retina_distance)

    return midpoint, goodness_of_fit

In [ ]:
"""EyelinkReader class."""


def normalize_eye(eye):
    norm_eye = eye - np.nanmean(eye, axis=0, keepdims=True)
    norm_eye /= np.nanstd(norm_eye, axis=0, keepdims=True)
    return norm_eye


def transform_between_eyes(eye_source, eye_target):
    keep_inds = np.logical_and(
        np.any(np.isfinite(eye_source), axis=1),
        np.any(np.isfinite(eye_target), axis=1),
    )

    keep_eye_source = eye_source[keep_inds]
    keep_eye_target = eye_target[keep_inds]

    reg = sklearn_linear_model.LinearRegression()
    reg.fit(keep_eye_source, keep_eye_target)
    pred_eye_target = reg.intercept_ + np.dot(eye_source, reg.coef_)

    return reg, pred_eye_target


class EyelinkReader:
    COLUMN_NAMES = [
        "time",
        "left_x",
        "left_y",
        "left_pupil",
        "right_x",
        "right_y",
        "right_pupil",
    ]

    def __init__(self, eyelink_filename):
        self.eyelink_filename = eyelink_filename
        self.dataframe = self()

    def _process_string(self, s):
        if s == ".":
            return np.nan
        else:
            return float(s)

    def __call__(self):
        """Return dataframe."""

        # Read EyeLink asc file
        print("Reading file %s..." % self.eyelink_filename)
        f = open(self.eyelink_filename, "r")
        file_text = f.read().splitlines(True)
        file_text = list(filter(None, file_text))
        file_text = np.array(file_text)
        f.close()

        # Find lines with data samples
        print("Finding lines with data samples")
        sample_lines = []
        for i, text in enumerate(file_text):
            if text != "\n" and text.split()[0][0].isdigit():
                sample_lines.append(i)

        # Create dataframe
        print("Creating dataframe")
        start_line = sample_lines[0]
        end_line = sample_lines[-1]
        dataframe = pd.read_csv(
            self.eyelink_filename,
            skiprows=start_line,
            nrows=end_line - start_line,
            delim_whitespace=True,
            index_col=False,
            names=EyelinkReader.COLUMN_NAMES,
            usecols=range(7),
        )

        for column in dataframe.columns:
            dataframe[column] = [
                self._process_string(x) for x in dataframe[column].tolist()
            ]

        start_time = dataframe["time"].tolist()[0]
        dataframe["time"] = (dataframe["time"] - start_time) / 1000.0

        time = dataframe["time"]
        print(f"Start time: {np.min(time)}")
        print(f"End time: {np.max(time)}")

        return dataframe

    def get_eye_pos(self, sample_rate_ms=2.0):
        # Check sample rate
        df_time = np.array(self.dataframe["time"])
        delta_df_time = df_time[1:] - df_time[:-1]
        if not np.allclose(delta_df_time, sample_rate_ms / 1000, atol=0.0001):
            raise ValueError(f"Sample rate is not always {sample_rate_ms}")

        # Get eye position and velocities
        left_eye = np.stack(
            [self.dataframe["left_x"], self.dataframe["left_y"]],
            axis=1,
        )
        right_eye = np.stack(
            [self.dataframe["right_x"], self.dataframe["right_y"]],
            axis=1,
        )

        bad_inds = np.logical_or(
            np.any(np.isnan(left_eye), axis=1),
            np.any(np.isnan(right_eye), axis=1),
        )
        left_eye[bad_inds] = np.nan
        right_eye[bad_inds] = np.nan

        left_eye = normalize_eye(left_eye)
        right_eye = normalize_eye(right_eye)

        _, left_eye = transform_between_eyes(left_eye, right_eye)

        return left_eye, right_eye

    def get_time_and_position(self):
        left_eye, right_eye = self._get_eye_pos()
        t = np.array(self.dataframe["time"])
        return t, left_eye, right_eye

    def get_smooth_eye_pos(self, kernel_size_ms, sample_rate_ms=2.0):
        # Get eye velocities
        left_eye, right_eye = self.get_eye_pos(sample_rate_ms=sample_rate_ms)

        # Get kernel
        kernel_size = int(kernel_size_ms // sample_rate_ms)
        linspace = np.linspace(0, 1, kernel_size)
        kernel = np.concatenate([linspace[:-1], linspace[::-1]])
        kernel /= np.sum(kernel)

        # Smooth eye position
        smooth_left = np.stack(
            [
                np.convolve(left_eye[:, i], kernel, mode="same")
                for i in range(2)
            ],
            axis=1,
        )
        smooth_right = np.stack(
            [
                np.convolve(right_eye[:, i], kernel, mode="same")
                for i in range(2)
            ],
            axis=1,
        )

        return smooth_left, smooth_right

In [ ]:
"""Functions for computing eye velocity and pursuit."""


def _get_smooth_eye_vel(eyelink_reader, kernel_size_ms, sample_rate_ms=2.0):
    # Get eye velocities
    left_eye, right_eye = eyelink_reader.get_eye_pos()
    left_v = left_eye[1:] - left_eye[:-1]
    right_v = right_eye[1:] - right_eye[:-1]

    # Get kernel
    kernel_size = int(kernel_size_ms // sample_rate_ms)
    linspace = np.linspace(0, 1, kernel_size)
    kernel = np.concatenate([linspace[:-1], linspace[::-1]])
    kernel /= np.sum(kernel)

    # Smooth and norm eye velocities
    smooth_left_v = np.stack(
        [np.convolve(left_v[:, i], kernel, mode="same") for i in range(2)],
        axis=1,
    )
    smooth_right_v = np.stack(
        [np.convolve(right_v[:, i], kernel, mode="same") for i in range(2)],
        axis=1,
    )
    smooth_left_vel = np.linalg.norm(smooth_left_v, axis=1)
    smooth_right_vel = np.linalg.norm(smooth_right_v, axis=1)

    return smooth_right_vel, smooth_left_vel


def _get_saccade_state(
    eyelink_reader,
    kernel_size_ms=30,
    threshold_vel=0.015,
    buffer_ms=80,
    sample_rate_ms=2.0,
):
    smooth_vel, _ = _get_smooth_eye_vel(
        eyelink_reader, kernel_size_ms=kernel_size_ms
    )
    saccade_state = smooth_vel > threshold_vel

    buffer_steps = int(buffer_ms // sample_rate_ms)
    boxcar_kernel = np.ones(buffer_steps)
    conv = np.convolve(saccade_state, boxcar_kernel, mode="same")
    saccade_state_buffer = conv >= 1

    return saccade_state_buffer


def _get_fixation_state(
    eyelink_reader,
    kernel_size_ms=100,
    threshold_vel=0.0015,
    buffer_ms=120,
    sample_rate_ms=2.0,
):
    smooth_vel, _ = _get_smooth_eye_vel(
        eyelink_reader, kernel_size_ms=kernel_size_ms
    )
    fixation_state = smooth_vel < threshold_vel

    buffer_steps = int(buffer_ms // sample_rate_ms)
    boxcar_kernel = np.ones(buffer_steps)
    conv = np.convolve(fixation_state, boxcar_kernel, mode="same")
    fixation_state_buffer = conv >= 1

    return fixation_state_buffer


def _get_untracked_state(eyelink_reader):
    smooth_vel, _ = _get_smooth_eye_vel(eyelink_reader, kernel_size_ms=10)
    untracked_state = np.isnan(smooth_vel)
    return untracked_state


def get_pursuit_state(eyelink_reader):
    saccade_state = _get_saccade_state(eyelink_reader)
    fixation_state = _get_fixation_state(eyelink_reader)
    untracked_state = _get_untracked_state(eyelink_reader)
    pursuit_state = (
        (1 - saccade_state) * (1 - fixation_state) * (1 - untracked_state)
    )

    # Hand-crafted pursuit state
    pursuit_state = np.zeros_like(pursuit_state)
    pursuit_intervals_sec = [
        (7.5, 9.5),
        (14.5, 15.2),
        (21, 21.9),
        (40, 41.5),
    ]
    for interval in pursuit_intervals_sec:
        pursuit_state[int(500 * interval[0]) : int(500 * interval[1])] = 1
    pursuit_state = pursuit_state.astype(bool)

    return pursuit_state


def plot_eye_state(eyelink_reader, time_range, sample_rate_ms=2):
    index_range = [
        int(1000 * time_range[0] / sample_rate_ms),
        int(1000 * time_range[1] / sample_rate_ms),
    ]

    saccade_state = _get_saccade_state(eyelink_reader)
    fixation_state = _get_fixation_state(eyelink_reader)
    pursuit_state = get_pursuit_state(eyelink_reader)
    pursuit_state = pursuit_state[index_range[0] : index_range[1]]
    pursuit_state_inds = np.argwhere(pursuit_state)[:, 0]
    left_eye, right_eye = eyelink_reader.get_eye_pos()
    left_vel, right_vel = _get_smooth_eye_vel(
        eyelink_reader, kernel_size_ms=30
    )
    t = np.array(eyelink_reader.dataframe["time"])
    t = t[index_range[0] : index_range[1]]

    fig, axes = plt.subplots(1, 3, figsize=(12, 3))
    axes[0].plot(t, left_eye[index_range[0] : index_range[1]], c="b")
    axes[0].plot(t, right_eye[index_range[0] : index_range[1]], c="r")
    pursuit_y = len(pursuit_state_inds) * [2.5]
    axes[0].scatter(t[pursuit_state_inds], pursuit_y, c="gray", s=10)
    axes[0].set_title("Eye Position")

    axes[1].plot(t, left_vel[index_range[0] : index_range[1]], c="b")
    axes[1].plot(t, right_vel[index_range[0] : index_range[1]], c="r")
    axes[1].set_title("Smooth Eye Velocity")
    axes[1].set_ylim([0, 0.04])

    axes[2].plot(
        t,
        saccade_state[index_range[0] : index_range[1]],
        c="r",
        label="saccade",
    )
    axes[2].plot(
        t,
        fixation_state[index_range[0] : index_range[1]],
        c="b",
        label="fixation",
    )
    axes[2].set_title("state")
    axes[2].legend(loc="center", bbox_to_anchor=(1.2, 0.5))

In [ ]:
_EYELINK_FILE = "../lalo_day_3/eyelink_files/l_d3_t2.asc"

eyelink_reader = EyelinkReader(_EYELINK_FILE)
pursuit_seconds = np.sum(get_pursuit_state(eyelink_reader) / 500)
print(f"Pursuit seconds = {pursuit_seconds}")

In [ ]:
start_sec = 0
plot_eye_state(eyelink_reader, time_range=(start_sec, start_sec + 10))

In [ ]:
"""Get gaze points."""


def get_gaze_points(
    eyelink_reader,
    theta=0.0,
    inward_skew=0.01,
    pupil_sep=10,
    retina_z=3,
    plot=True,
):
    smooth_left, smooth_right = eyelink_reader.get_smooth_eye_pos(
        kernel_size_ms=6
    )
    midpoints, goodness_of_fit = _compute_focus_points(
        smooth_right,
        smooth_left,
        retina_z=retina_z,
        theta=theta,
        pupil_sep=pupil_sep,
        inward_skew=inward_skew,
    )

    if plot:
        pursuit_state = get_pursuit_state(eyelink_reader).astype(bool)
        midpoints_pursuit = midpoints[:-1][pursuit_state]

        fig = plt.figure(figsize=(8, 4))
        ax = fig.add_subplot(1, 2, 1, projection="3d")
        ax.scatter(midpoints[:, 0], midpoints[:, 1], midpoints[:, 2], c="b")
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        ax.set_title("All")

        ax = fig.add_subplot(1, 2, 2, projection="3d")
        ax.scatter(
            midpoints_pursuit[:, 0],
            midpoints_pursuit[:, 1],
            midpoints_pursuit[:, 2],
            c="b",
        )
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        ax.set_zlabel("z")
        ax.set_title("Pursuit")

    return midpoints, goodness_of_fit


gaze_points, goodness_of_fit = get_gaze_points(
    eyelink_reader, inward_skew=0.05, retina_z=10
)

In [ ]:
"""OptitrackReader class."""


class OptitrackReader:
    COLUMN_NAMES = [
        "time",
        "x",
        "y",
        "z",
    ]

    def __init__(self, optitrack_filename):
        self.optitrack_filename = optitrack_filename
        self.dataframe = self()

    def _process_string(self, s):
        if s == ".":
            return np.nan
        else:
            return float(s)

    def __call__(self):
        """Return dataframe."""

        # Read optitrack file
        print("Reading file %s..." % self.optitrack_filename)
        f = open(self.optitrack_filename, "r")
        file_text = f.read().splitlines(True)
        file_text = list(filter(None, file_text))
        file_text = np.array(file_text)
        f.close()

        # Find lines with data samples
        print("Finding lines with data samples")
        sample_lines = []
        for i, text in enumerate(file_text):
            if text != "\n" and text.split()[0][0].isdigit():
                sample_lines.append(i)

        # Create dataframe
        print("Creating dataframe")
        start_line = sample_lines[0]
        end_line = sample_lines[-1]
        dataframe = pd.read_csv(
            self.optitrack_filename,
            sep=",",
            skiprows=start_line,
            nrows=end_line - start_line,
            index_col=False,
            names=OptitrackReader.COLUMN_NAMES,
            usecols=range(1, 5),
        )

        for column in dataframe.columns:
            dataframe[column] = [
                self._process_string(x) for x in dataframe[column].tolist()
            ]

        time = dataframe["time"]
        print(f"Start time: {np.min(time)}")
        print(f"End time: {np.max(time)}")

        return dataframe

    def get_time_and_position(self):
        x = np.array(self.dataframe["x"])
        y = np.array(self.dataframe["y"])
        z = np.array(self.dataframe["z"])
        t = np.array(self.dataframe["time"])
        position = np.stack([x, y, z], axis=1)
        return t, position

    def plot(self, time_range, sample_rate_ms=8.33333333):
        index_range = [
            int(1000 * time_range[0] / sample_rate_ms),
            int(1000 * time_range[1] / sample_rate_ms),
        ]

        t, position = self.get_time_and_position()
        t = t[index_range[0] : index_range[1]]
        position = position[index_range[0] : index_range[1]]
        position[np.isnan(position)] = 0.0

        fig, ax = plt.subplots(1, 1, figsize=(4, 3))
        ax.plot(t, position[:, 0], c="b")
        ax.plot(t, position[:, 1], c="r")
        ax.plot(t, position[:, 2], c="g")
        ax.set_title("Position")

In [ ]:
_OPTITRACK_FILE = "../lalo_day_3/optitrack_files/l_d3_t2.csv"

optitrack_reader = OptitrackReader(_OPTITRACK_FILE)
optitrack_df = optitrack_reader.dataframe

In [ ]:
optitrack_reader.plot(time_range=(0, 28))

In [ ]:
"""Regression functions."""


def get_regression_data(use_gaze_points=True):
    if use_gaze_points:
        pos_eye_full = gaze_points
    else:
        left_eye, right_eye = eyelink_reader.get_smooth_eye_pos(
            kernel_size_ms=10
        )
        pos_eye_full = np.concatenate([left_eye, right_eye], axis=1)

    # Get data
    t_eye = np.array(eyelink_reader.dataframe["time"])
    pursuit_state = get_pursuit_state(eyelink_reader).astype(bool)
    t_o, pos_o = optitrack_reader.get_time_and_position()
    print(f"Eye range: [{np.min(t_eye)}, {np.max(t_eye)}]")
    print(f"Optitrack range: [{np.min(t_o)}, {np.max(t_o)}]")

    # Resample eye data
    t_o_freq = 1.0 / 120
    t_eye_freq = 1.0 / 500
    eye_sub_inds = np.round(
        np.arange(0, t_eye[-1], t_o_freq) / t_eye_freq
    ).astype(int)
    t_eye = t_eye[eye_sub_inds]
    pos_eye = pos_eye_full[eye_sub_inds]
    pursuit_eye = pursuit_state[eye_sub_inds].astype(bool)

    return pos_eye, pos_o, pursuit_eye


def compute_regression_score(pos_o, pos_e, pursuit, eye_start_ind=0):
    # Create offset o_pursuit_state
    if eye_start_ind > 0:
        pos_o = pos_o[eye_start_ind:]
    elif eye_start_ind < 0:
        pos_e = pos_e[-eye_start_ind:]

    # Trim everything
    length = min(pos_o.shape[0], pos_e.shape[0], len(pursuit))
    pos_o = pos_o[:length]
    pos_e = pos_e[:length]
    pursuit = pursuit[:length]

    pursuit_pos_o = pos_o[pursuit]
    pursuit_pos_e = pos_e[pursuit]

    pursuit_length = min(pursuit_pos_o.shape[0], pursuit_pos_e.shape[0])
    pursuit_pos_o = pursuit_pos_o[:pursuit_length]
    pursuit_pos_e = pursuit_pos_e[:pursuit_length]

    good_inds = np.all(np.isfinite(pursuit_pos_o), axis=1) * np.all(
        np.isfinite(pursuit_pos_e), axis=1
    )
    pursuit_pos_o = pursuit_pos_o[good_inds]
    pursuit_pos_e = pursuit_pos_e[good_inds]

    reg = sklearn_linear_model.LinearRegression()
    reg.fit(pursuit_pos_e, pursuit_pos_o)
    pred_pursuit_pos_o = reg.predict(pursuit_pos_e)

    return reg, pursuit_pos_e, pursuit_pos_o, pred_pursuit_pos_o

In [ ]:
"""Compute and plot regression."""

pos_eye, pos_o, pursuit_eye = get_regression_data(use_gaze_points=True)

scores = []
eye_start_inds = range(-100, 100)
for eye_start_ind in eye_start_inds:
    reg, pursuit_pos_e, pursuit_pos_o, pred_pursuit_pos_o = (
        compute_regression_score(
            pos_o, pos_eye, pursuit_eye, eye_start_ind=eye_start_ind
        )
    )
    scores.append(reg.score(pursuit_pos_e, pursuit_pos_o))

fig, ax = plt.subplots(figsize=(3, 3))
ax.plot(scores)

best_eye_start_ind = eye_start_inds[np.argmax(scores)]
reg, pursuit_pos_e, pursuit_pos_o, pred_pursuit_pos_o = (
    compute_regression_score(
        pos_o, pos_eye, pursuit_eye, eye_start_ind=best_eye_start_ind
    )
)
fig, axes = plt.subplots(1, 3, figsize=(6, 2))
for i in range(3):
    axes[i].scatter(pursuit_pos_o[:, i], pred_pursuit_pos_o[:, i])

# Plot full prediction

fig = plt.figure(figsize=(12, 3))


def _add_subplot(ax_index, data, title):
    ax = fig.add_subplot(1, 4, ax_index, projection="3d")
    ax.scatter(data[:, 0], data[:, 1], data[:, 2], c="b")
    ax.set_xlabel("x")
    ax.set_ylabel("y")
    ax.set_zlabel("z")
    ax.set_title(title)


# pred_full = reg.intercept_[None] + np.dot(pos_eye_full, reg.coef_.T)
# _add_subplot(1, pred_full, 'Gaze Full')
_add_subplot(1, pos_o, "Optitrack Full")
_add_subplot(2, pred_pursuit_pos_o, "Gaze Pursuit")
_add_subplot(3, pursuit_pos_o, "Optitrack Pursuit")